# Retraining with different MLP activations

The ReLU model learns addition mod $N$ by storing each number as an angle on a few circles
and letting the MLP multiply rotations. This notebook retrains the same transformer from
scratch with the ReLU swapped for activations that look more like integer arithmetic on a
number line, and compares the training curves.

Activations compared (all with a `period` parameter, default $N$):

| name | hidden activation | periodic? | differentiable? |
|---|---|---|---|
| `relu` | $\max(x, 0)$ | no | kink at 0 |
| `sawtooth` | $x \bmod P$ | yes | derivative 1 a.e., jumps at multiples of $P$ |
| `smooth_staircase` | $x - \tfrac{P}{2\pi}\sin\tfrac{2\pi x}{P}$ | **no** | everywhere |
| `fourier_sawtooth` | $\tfrac{P}{2} - \tfrac{P}{\pi}\sum_{k=1}^{K}\tfrac{1}{k}\sin\tfrac{2\pi k x}{P}$ | yes | everywhere |

A correction to an earlier suggestion: $x - \frac{P}{2\pi}\sin\frac{2\pi x}{P}$ is *not* a
smoothed sawtooth. Its derivative is $1 - \cos(2\pi x / P) \ge 0$, so it is a monotone
staircase that flattens near multiples of $P$ and never wraps. It is kept here as a control.
The truncated Fourier series of $x \bmod P$ is the smooth, periodic version; $K$ controls
how sharp the seam is.

The hypothesis under test: the number-line solution (embed $a$ as a magnitude, sum, reduce
mod $N$) is expressible with a sawtooth but is not a basin of the loss, because the
sawtooth's gradient is 1 everywhere and the wraps are invisible to backpropagation.

In [ ]:
import json
import math
import time
import dataclasses
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from config import default_config as conf
from data import generate_data_tensor
from loss import loss_fn
from model import model
from embedding import embedding
from attention import attention
from unembedding import unembedding

N = conf.modulus
torch.manual_seed(0)
X, Y = generate_data_tensor(conf)              # all N**2 pairs, one-hot, shuffled
n_train = round(0.8 * len(X))
X_train, Y_train = X[:n_train], Y[:n_train]
X_test,  Y_test  = X[n_train:], Y[n_train:]
print(f"modulus {N}: {len(X_train)} training pairs, {len(X_test)} held out")

In [ ]:
@torch.no_grad()
def eval_metrics(net, X, Y):
    "Answer-token loss and accuracy."
    logits = net(X)
    loss = loss_fn(Y, logits).item()
    acc = (logits[:, -1].argmax(-1) == Y[:, -1].argmax(-1)).float().mean().item()
    return loss, acc


def train_and_record(net, steps, log_every=25):
    "Full-batch training for `steps` optimizer steps; returns a dict of metric histories."
    hist = {k: [] for k in ("step", "train_loss", "test_loss", "train_acc", "test_acc")}
    for step in range(steps + 1):
        if step % log_every == 0 or step == steps:
            (tr_loss, tr_acc) = eval_metrics(net, X_train, Y_train)
            (te_loss, te_acc) = eval_metrics(net, X_test, Y_test)
            for key, val in zip(hist, (step, tr_loss, te_loss, tr_acc, te_acc)):
                hist[key].append(val)
        if step < steps:
            net.train(X_train, Y_train)
    return hist


COLORS = ["#2a78d6", "#eb6834", "#3aa655", "#8e5bd0", "#c9a227", "#4a4a4a"]

def plot_histories(histories, key, title=None):
    "histories: {name: hist}. key: 'loss' or 'acc'. Solid = train, dashed = test."
    fig, ax = plt.subplots(figsize=(8, 3.8))
    for (name, h), color in zip(histories.items(), COLORS):
        ax.plot(h["step"], h[f"train_{key}"], color=color, lw=1.5, label=f"{name}, train")
        ax.plot(h["step"], h[f"test_{key}"], color=color, lw=1.5, ls="--", label=f"{name}, test")
    ax.set(xlabel="optimizer step", ylabel=key, title=title,
           yscale="log" if key == "loss" else "linear")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    plt.show()

## The activations

Each factory returns a function `hidden = activation(pre_activation)` that replaces
`torch.relu` in the MLP.

In [ ]:
def relu():
    return torch.relu


def sawtooth(period):
    "Hard sawtooth x mod period. Autograd passes the gradient straight through (derivative 1)."
    return lambda x: torch.remainder(x, period)


def smooth_staircase(period):
    "x - (period/2π) sin(2πx/period). Smooth and monotone; NOT periodic, so it never reduces mod period."
    return lambda x: x - (period / (2 * math.pi)) * torch.sin(2 * math.pi * x / period)


def fourier_sawtooth(period, harmonics):
    "Truncated Fourier series of x mod period. Periodic and smooth; larger `harmonics` sharpens the seam."
    k = torch.arange(1, harmonics + 1, dtype=torch.float32)

    def f(x):
        phases = 2 * math.pi * x.unsqueeze(-1) * k / period        # (..., harmonics)
        return period / 2 - (period / math.pi) * (torch.sin(phases) / k).sum(-1)
    return f


# Look at them before training with them.
xs = torch.linspace(-1.5 * N, 1.5 * N, 2000)
fig, ax = plt.subplots(figsize=(8, 3.6))
for (name, f), color in zip({"relu": relu(),
                             f"sawtooth, P={N}": sawtooth(N),
                             f"smooth staircase, P={N}": smooth_staircase(N),
                             f"fourier sawtooth, P={N}, K=5": fourier_sawtooth(N, 5)}.items(), COLORS):
    ax.plot(xs, f(xs), color=color, lw=1.5, label=name)
ax.set(xlabel="pre-activation x", ylabel="hidden activation", ylim=(-5, 1.5 * N))
ax.axvspan(-3, 3, color="#999", alpha=0.15, label="typical scale at initialization")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
plt.show()

## The model

Same weights, optimizer, and forward pass as `model`; only the MLP nonlinearity changes.
Each instance gets its own TensorBoard directory so the logs don't collide.

In [ ]:
class custom_mlp_model(model):
    "The one-layer transformer with a pluggable MLP activation."

    def __init__(self, c, activation):
        super().__init__(c)
        self.activation = activation

    def forward(self, x):
        x1 = attention(self, embedding(self, x))
        hidden = self.activation(x1 @ self.W_i + self.b_i)
        return unembedding(self, hidden @ self.W_o + self.b_o)


def fresh_model(name, activation, seed):
    "Seed right before construction so every variant starts from identical weights."
    torch.manual_seed(seed)
    logdir = f"runs/{name.replace(' ', '_').replace(',', '').replace('=', '')}"
    return custom_mlp_model(dataclasses.replace(conf, logdir=logdir), activation)

## Settings

`STEPS = 3000` is roughly ten minutes per model on a laptop CPU. Results are written to
`OUTPUT_DIR` (metric histories as JSON, weights as `.pt`) so they can be copied back and
analysed elsewhere.

In [ ]:
STEPS = 3000
LOG_EVERY = 25
SEED = 1
PERIOD = N
HARMONICS = 5
OUTPUT_DIR = Path("experiment_results")
OUTPUT_DIR.mkdir(exist_ok=True)

## Experiment 1: four activations, same period

Prediction: ReLU groks; the hard sawtooth with $P = N$ is the identity at initialization and
trains like a linear model, so it sits at chance on the held-out set; the staircase is a
smooth near-identity and behaves like a worse ReLU; the Fourier sawtooth is the interesting
one, since its gradient can see the wraps.

In [ ]:
experiments = {
    "relu": relu(),
    f"sawtooth P={PERIOD}": sawtooth(PERIOD),
    f"smooth staircase P={PERIOD}": smooth_staircase(PERIOD),
    f"fourier sawtooth P={PERIOD} K={HARMONICS}": fourier_sawtooth(PERIOD, HARMONICS),
}

histories = {}
for name, activation in experiments.items():
    net = fresh_model(name, activation, SEED)
    t0 = time.time()
    histories[name] = train_and_record(net, STEPS, LOG_EVERY)
    final = histories[name]
    print(f"{name:40s} {time.time() - t0:6.0f} s   "
          f"train acc {final['train_acc'][-1]:.3f}   test acc {final['test_acc'][-1]:.3f}")
    torch.save(net.state_dict(), OUTPUT_DIR / f"{name.replace(' ', '_')}.pt")

with (OUTPUT_DIR / "activation_histories.json").open("w") as f:
    json.dump(histories, f)

In [ ]:
plot_histories(histories, "loss", f"answer-token loss, N = {N}")
plot_histories(histories, "acc", f"answer accuracy, N = {N}")

## Experiment 2: hard sawtooth, period sweep

The scale argument says the period has to be comparable to the pre-activation scale before
the sawtooth does anything. Sweep it downward and watch when (if) behaviour changes.

In [ ]:
PERIODS = [N, 8, 2, 0.5]

sweep_histories = {}
for period in PERIODS:
    name = f"sawtooth P={period}"
    net = fresh_model(name, sawtooth(period), SEED)
    t0 = time.time()
    sweep_histories[name] = train_and_record(net, STEPS, LOG_EVERY)
    final = sweep_histories[name]
    print(f"{name:40s} {time.time() - t0:6.0f} s   "
          f"train acc {final['train_acc'][-1]:.3f}   test acc {final['test_acc'][-1]:.3f}")

with (OUTPUT_DIR / "sawtooth_period_histories.json").open("w") as f:
    json.dump(sweep_histories, f)

plot_histories(sweep_histories, "loss", "hard sawtooth, period sweep: loss")
plot_histories(sweep_histories, "acc", "hard sawtooth, period sweep: accuracy")

## Reading the results

- **Test accuracy at chance with train accuracy climbing** is memorization. With train
  accuracy also stuck, the model is effectively linear.
- **Which representation did a successful model find?** Reload its weights into a
  `custom_mlp_model`, take the column-wise DFT of `W_E[:N]` as in `training.ipynb`, and check
  whether the embedding is still a few circles or has become a line (one dominant singular
  direction with entries growing linearly in the token).
- For a model that groks with a periodic activation, redo the 2D DFT of the hidden
  activations and see whether the energy is still on the diagonal $k_a = k_b$.